# Calculating Net Market Income 

In [9]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Configuración de rutas (asegúrate de que ROOT suba a la raíz)
ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from config.paths import RAW_DIR, PROCESSED_DIR

YEARS = [2018, 2024]


# Defining brackets

In [10]:
# Definimos los parámetros de la tabla ISR (Basado en tu código de Stata)
# Nota: En un proyecto real, podrías tener una tabla para 2018 y otra para 2024
TAX_BRACKETS = [
    {"li": 0.01,       "ls": 8952.49,    "cf": 0,          "tm": 0.0192},
    {"li": 8952.50,    "ls": 75984.55,   "cf": 171.88,     "tm": 0.064},
    {"li": 75984.56,   "ls": 133536.07,  "cf": 4461.94,    "tm": 0.1088},
    {"li": 133536.08,  "ls": 155229.80,  "cf": 10723.55,   "tm": 0.16},
    {"li": 155229.81,  "ls": 185852.57,  "cf": 14194.54,   "tm": 0.1792},
    {"li": 185852.58,  "ls": 374837.88,  "cf": 19682.13,   "tm": 0.2136},
    {"li": 374837.89,  "ls": 590795.99,  "cf": 60049.40,   "tm": 0.2352},
    {"li": 590796.00,  "ls": 1127926.84, "cf": 110842.74,  "tm": 0.30},
    {"li": 1127926.85, "ls": 1503902.46, "cf": 271981.99,  "tm": 0.32},
    {"li": 1503902.47, "ls": 4511707.37, "cf": 392294.17,  "tm": 0.34},
    {"li": 4511707.38, "ls": np.inf,     "cf": 1414947.85, "tm": 0.35},
]

In [11]:
def calculate_gross_income(df, brackets):
    """Calcula el ingreso bruto a partir del neto trimestral (Gross-up)"""
    df['ing_bruto'] = np.nan
    
    for b in brackets:
        # Aplicamos la fórmula inversa: Gross = (Net + CF - TM * LI) / (1 - TM)
        b_temp = (df['ing_tri_anual'] + b['cf'] - b['tm'] * b['li']) / (1 - b['tm'])
        
        # Máscara para identificar qué registros caen en este tramo
        mask = (df['ing_bruto'].isna()) & \
               (b_temp >= b['li']) & \
               (b_temp <= b['ls'])
        
        df.loc[mask, 'ing_bruto'] = b_temp
    
    return df

# Ciclo para procesar ambos años
for year in YEARS:
    print(f"📊 Procesando año {year}...")
    
    # 1. Cargar datos de ingresos
    file_path = RAW_DIR / f"ingresos_{year}.csv"
    df_ing = pd.read_csv(file_path)
    
    # 2. Filtrar por clave P001 (Sueldos y salarios)
    df_p001 = df_ing[df_ing['clave'] == 'P001'].copy()
    
    # 3. Aplicar el cálculo del ingreso bruto
    # Asegúrate de que 'ing_tri' es el nombre de la columna en tu CSV
    df_p001 = df_p001.rename(columns={'ing_tri': 'ing_tri_anual'}) 
    df_p001 = calculate_gross_income(df_p001, TAX_BRACKETS)
    
    # 4. Guardar avance
    output_path = PROCESSED_DIR / f"ingresos_brutos_{year}.csv"
    df_p001.to_csv(output_path, index=False)
    print(f"   ✅ Guardado en: {output_path.name}")

print("\n✨ Primer paso del CEQ completado.")

📊 Procesando año 2018...
   ✅ Guardado en: ingresos_brutos_2018.csv
📊 Procesando año 2024...
   ✅ Guardado en: ingresos_brutos_2024.csv

✨ Primer paso del CEQ completado.


In [19]:
a = RAW_DIR / f"ingresos_2018.csv"
b = pd.read_csv(a)
b

b["clave"].value_counts

<bound method IndexOpsMixin.value_counts of 0         P032
1         P001
2         P001
3         P044
4         P063
          ... 
348482    P001
348483    P040
348484    P044
348485    P040
348486    P044
Name: clave, Length: 348487, dtype: str>

In [12]:
# 1. Cargar los DataFrames procesados
df_2018 = pd.read_csv(PROCESSED_DIR / "ingresos_brutos_2018.csv")
df_2024 = pd.read_csv(PROCESSED_DIR / "ingresos_brutos_2024.csv")

# 2. Mostrar un resumen rápido de 2018
print(f"{'='*20} ENIGH 2018 {'='*20}")
print(f"Registros procesados (P001): {len(df_2018)}")
display(df_2018[['folioviv', 'foliohog', 'numren', 'ing_tri_anual', 'ing_bruto']].head())

# 3. Mostrar un resumen rápido de 2024
print(f"\n{'='*20} ENIGH 2024 {'='*20}")
print(f"Registros procesados (P001): {len(df_2024)}")
display(df_2024[['folioviv', 'foliohog', 'numren', 'ing_tri_anual', 'ing_bruto']].head())

# 4. Comparación estadística rápida de la columna generada
summary_comparison = pd.concat([
    df_2018['ing_bruto'].describe().rename('2018'),
    df_2024['ing_bruto'].describe().rename('2024')
], axis=1)  

print("\n--- Resumen Estadístico del Ingreso Bruto (Market Income) ---")
display(summary_comparison)

==================== ENIGH 2018 ====================
Registros procesados (P001): 88886


,folioviv,foliohog,numren,ing_tri_anual,ing_bruto
0,100013601,1,1,29508.19,31097.339744
1,100013601,1,3,23606.55,24792.168803
2,100013603,1,1,110163.93,119343.301023
3,100013603,1,2,23606.55,24792.168803
4,100013606,1,3,8852.45,9029.241453



==================== ENIGH 2024 ====================
Registros procesados (P001): 106397


,folioviv,foliohog,numren,ing_tri_anual,ing_bruto
0,100001901,1,1,48952.17,51870.822650
1,100001901,1,2,29347.82,30926.004274
2,100001902,1,1,46956.52,49738.717949
3,100001902,1,3,29347.82,30926.004274
4,100001904,1,1,5869.56,5984.461468



--- Resumen Estadístico del Ingreso Bruto (Market Income) ---


,2018,2024
count,8.888600e+04,1.063970e+05
mean,1.864511e+04,3.025013e+04
std,2.044863e+04,2.826258e+04
min,1.245902e+01,2.492843e+01
25%,8.424031e+03,1.587583e+04
50%,1.487111e+04,2.479217e+04
75%,2.277384e+04,3.719691e+04
max,1.279503e+06,1.576070e+06


# Imputation of IVA

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

# Configuración de prefijos IVA y códigos de informalidad por año
YEAR_CONFIG = {
    2018: {
        # Prefijos: C (Limpieza), D (Cuidados), F (Comunicaciones/Combustible), 
        # H (Vestido), I (Muebles), K (Enseres), L (Esparcimiento), M (Transporte), N (Otros)
        "iva_prefixes": ('C', 'D', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N'),
        "informal_locs": [1, 2, 3, 17], # Códigos 2018
        "clave_len": 4
    },
    2024: {
        # Prefijos COICOP: 02 (Alcohol), 03 (Vestido), 05 (Muebles), 07 (Transporte), 
        # 08 (Comunicación), 09 (Recreación), 11 (Restaurantes), 12 (Otros)
        "iva_prefixes": ('012', '02', '03', '05', '07', '08', '09', '11', '12'),
        "informal_locs": [1, 2, 3, 17], # Generalmente se mantienen, verificar en FD
        "clave_len": 6
    }
}

def fix_and_calculate_iva(year, raw_dir):
    print(f"🛠️  Procesando ENIGH {year}...")
    
    config = YEAR_CONFIG[year]
    
    # 1. Carga de datos
    df_gastos = pd.read_csv(raw_dir / f"gastoshogar_{year}.csv", 
                            dtype={'folioviv': str, 'foliohog': str, 'clave': str}, 
                            low_memory=False)
    df_conc = pd.read_csv(raw_dir / f"concentradohogar_{year}.csv", 
                          dtype={'folioviv': str, 'foliohog': str}, 
                          low_memory=False)

    # 2. Procesamiento de Gastos
    df_gastos['gasto_tri'] = pd.to_numeric(df_gastos['gasto_tri'], errors='coerce').fillna(0)
    
    # Filtrar solo gastos monetarios (G1)
    df_gastos = df_gastos[df_gastos['tipo_gasto'] == 'G1'].copy()
    
    # Identificar si lleva IVA basado en los prefijos del año correspondiente
    df_gastos['lleva_iva'] = df_gastos['clave'].str.startswith(config["iva_prefixes"]).astype(int)
    
    # Identificar formalidad 
    df_gastos['es_formal'] = (~df_gastos['lugar_comp'].isin(config["informal_locs"])).astype(int)
    
    # Cálculo del IVA por partida: Gasto * (0.16 / 1.16)
    # $Factor_{IVA} = \frac{0.16}{1.16} \approx 0.137931$ 
    factor_iva = 0.137931034
    df_gastos['iva_partida'] = np.where((df_gastos['lleva_iva'] == 1) & (df_gastos['es_formal'] == 1),
                                        df_gastos['gasto_tri'] * factor_iva, 0)
    
    # Agrupar IVA por hogar
    iva_hogar = df_gastos.groupby(['folioviv', 'foliohog'])['iva_partida'].sum().reset_index()
    iva_hogar.rename(columns={'iva_partida': 'IVA_HH'}, inplace=True)

    # 3. Preparar Concentrado
    cols_num = ['gasto_mon', 'alimentos', 'limpieza', 'cuidados', 'ing_cor', 'factor']
    for col in cols_num:
        df_conc[col] = pd.to_numeric(df_conc[col], errors='coerce').fillna(0)
    
    # Base de gasto estructural (excluyendo lo básico que suele ser tasa 0% o exento)
    df_conc['Gasto_HH_tri'] = df_conc['gasto_mon'] - (df_conc['alimentos'] + df_conc['limpieza'] + df_conc['cuidados'])

    # 4. Merge y cálculo de tasa efectiva
    df_final = pd.merge(df_conc, iva_hogar, on=['folioviv', 'foliohog'], how='left').fillna(0)
    
    # Cálculo del IVA atribuible al ingreso (proporcional)
    # Evitamos división por cero con np.where
    df_final['IVA_HH_A'] = np.where(df_final['Gasto_HH_tri'] > 0,
                                    (df_final['IVA_HH'] / df_final['Gasto_HH_tri']) * df_final['ing_cor'],
                                    0)
    
    return df_final.copy()

# Ejecución del Loop
results = {}
for year in [2018, 2024]:
    df_result = fix_and_calculate_iva(year, RAW_DIR)
    
    # Cálculo de Deciles
    df_result = df_result.sort_values("ing_cor")
    df_result["cumsum_factor"] = df_result["factor"].cumsum()
    df_result["decil"] = pd.qcut(df_result["cumsum_factor"], 10, labels=range(1, 11))
    
    # Resumen ponderado
    summary = df_result.groupby("decil").agg(
        ingreso_medio=("ing_cor", lambda x: np.average(x, weights=df_result.loc[x.index, "factor"])),
        iva_medio=("IVA_HH_A", lambda x: np.average(x, weights=df_result.loc[x.index, "factor"])),
        hogares_representados=("factor", "sum")
    ).reset_index()
    
    summary["carga_iva_pct"] = (summary["iva_medio"] / summary["ingreso_medio"]) * 100
    results[year] = summary

    print(f"\n{'='*20} ENIGH {year} {'='*20}")
    display(summary.style.format({
        "ingreso_medio": "${:,.2f}", "iva_medio": "${:,.2f}", 
        "carga_iva_pct": "{:.2f}%", "hogares_representados": "{:,.0f}"
    }))

🛠️  Procesando ENIGH 2018...

==================== ENIGH 2018 ====================


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_71351/3005975830.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_conc['Gasto_HH_tri'] = df_conc['gasto_mon'] - (df_conc['alimentos'] + df_conc['limpieza'] + df_conc['cuidados'])
/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_71351/3005975830.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['IVA_HH_A'] = np.where(df_final['Gasto_HH_tri'] > 0,


,decil,ingreso_medio,iva_medio,hogares_representados,carga_iva_pct
0,1,"$8,716.22","$1,658.35","3,077,839",19.03%
1,2,"$15,291.60","$2,251.26","3,127,662",14.72%
2,3,"$20,324.76","$2,223.45","3,221,401",10.94%
3,4,"$25,314.82","$2,806.68","3,299,920",11.09%
4,5,"$30,614.86","$3,098.35","3,412,828",10.12%
5,6,"$36,961.89","$3,629.89","3,460,309",9.82%
6,7,"$44,727.99","$4,388.86","3,501,973",9.81%
7,8,"$55,585.77","$5,446.05","3,575,937",9.80%
8,9,"$73,631.94","$8,816.83","3,682,880",11.97%
9,10,"$156,510.85","$18,790.49","4,039,766",12.01%


🛠️  Procesando ENIGH 2024...

==================== ENIGH 2024 ====================


/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_71351/3005975830.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_conc['Gasto_HH_tri'] = df_conc['gasto_mon'] - (df_conc['alimentos'] + df_conc['limpieza'] + df_conc['cuidados'])
/var/folders/lw/wsksmjcd29scklkcylgwjjlr0000gn/T/ipykernel_71351/3005975830.py:71: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_final['IVA_HH_A'] = np.where(df_final['Gasto_HH_tri'] > 0,


,decil,ingreso_medio,iva_medio,hogares_representados,carga_iva_pct
0,1,"$15,688.53","$5,014.86","3,283,413",31.97%
1,2,"$26,482.16","$3,675.52","3,532,617",13.88%
2,3,"$34,539.90","$4,416.71","3,634,153",12.79%
3,4,"$42,452.03","$5,475.26","3,746,813",12.90%
4,5,"$51,059.62","$6,283.88","3,861,828",12.31%
5,6,"$60,895.15","$7,609.87","3,949,553",12.50%
6,7,"$72,846.22","$9,077.25","3,922,847",12.46%
7,8,"$89,292.18","$12,463.10","4,055,830",13.96%
8,9,"$115,561.16","$14,573.56","4,229,375",12.61%
9,10,"$220,959.36","$34,181.70","4,613,801",15.47%


In [3]:
import graphviz

# =====================================================
# CEQ FISCAL INCIDENCE FRAMEWORK - MEXICO (2018-2024)
# CONSISTENT WITH ACADEMIC JOURNAL STYLE
# =====================================================

dot = graphviz.Digraph(comment='CEQ Mexico Methodological Framework')

# Global Settings
dot.attr(
    rankdir='LR',
    splines='ortho',
    nodesep='0.6',
    ranksep='0.8',
    fontname='Times New Roman',
    fontsize='11'
)

# Color Scheme
color_process = '#6C757D'   # Processing/Steps (Gray)
color_income = '#2F5D8C'    # Main Income Concepts (Blue)
color_box = '#E9ECEF'       # Background nodes
color_cluster = 'black'

# -----------------------------------------------------
# 1. INPUTS & DATA SOURCES
# -----------------------------------------------------
with dot.subgraph(name='cluster_sources') as src:
    src.attr(label='Data Sources & Parameters', style='rounded', color=color_cluster)
    
    src.node('ENIGH', 'ENIGH 2018 & 2024\n(Microdata: Ingresos/Gastos)', shape='box', style='filled', fillcolor=color_box)
    src.node('Statutory', 'Statutory Tax Rules\n(ISR Tables & VAT Rates)', shape='box', style='filled', fillcolor=color_box)
    src.node('Accounts', 'National Accounts\n(Public Spending Totals)', shape='box', style='filled', fillcolor=color_box)

# -----------------------------------------------------
# 2. COMPUTATIONAL MODULES (Your Python Logic)
# -----------------------------------------------------
with dot.subgraph(name='cluster_modules') as mod:
    mod.attr(label='Fiscal Estimation Modules (Python)', style='rounded', color=color_cluster)
    
    mod.node('ISR', 'Personal Income Tax (ISR)\n• Gross-up Procedure\n• Taxable Base (bg) Simulation', 
             shape='box', style='filled', fillcolor=color_process, fontcolor='white')
    
    mod.node('VAT', 'Value Added Tax (VAT)\n• Direct Identification\n• Informality Adjustment\n• Integrated Factor (0.1379)', 
             shape='box', style='filled', fillcolor=color_process, fontcolor='white')

# -----------------------------------------------------
# 3. CEQ INCOME CONCEPTS (The Core Chain)
# -----------------------------------------------------
# Based on [cite: 390, 679-721]
with dot.subgraph(name='cluster_concepts') as con:
    con.attr(label='CEQ Income Concepts', style='rounded', color=color_cluster)
    
    con.node('MI', 'Market Income (Im)\n(Wages + Capital + Pensions)', shape='ellipse', style='filled', fillcolor=color_income, fontcolor='white')
    con.node('NMI', 'Net Market Income (In)\n(Im - Direct Taxes)', shape='ellipse', style='filled', fillcolor=color_income, fontcolor='white')
    con.node('DI', 'Disposable Income (Id)\n(In + Direct Transfers)', shape='ellipse', style='filled', fillcolor=color_income, fontcolor='white')
    con.node('PFI', 'Post-Fiscal Income (Ipf)\n(Id - Indirect Taxes)', shape='ellipse', style='filled', fillcolor=color_income, fontcolor='white')
    con.node('FI', 'Final Income (If)\n(Ipf + In-kind Transfers)', shape='ellipse', style='filled', fillcolor=color_income, fontcolor='white')

# -----------------------------------------------------
# 4. OUTPUTS & ANALYSIS
# -----------------------------------------------------
dot.node('Results', 'Redistributive Impact\n• Gini Coefficient\n• Headcount Ratio (Poverty)\n• Effectiveness Indicator', 
         shape='box', style='filled', fillcolor=color_box)

# -----------------------------------------------------
# EDGES (Defining the Flow)
# -----------------------------------------------------
# Connections from Sources to Modules
dot.edge('ENIGH', 'ISR')
dot.edge('Statutory', 'ISR')
dot.edge('ENIGH', 'VAT')

# Connections to Income Concepts
dot.edge('ISR', 'MI', label='Pre-tax Base')
dot.edge('MI', 'NMI', label='Tax Deduction')
dot.edge('NMI', 'DI', label='+ Social Programs')
dot.edge('VAT', 'PFI', label='VAT Burden')
dot.edge('DI', 'PFI')
dot.edge('PFI', 'FI', label='+ Health/Education')

# Final result
dot.edge('FI', 'Results')

# -----------------------------------------------------
# RENDER
# -----------------------------------------------------
dot.render('CEQ_Mexico_Framework', format='pdf', view=True, cleanup=True)

'CEQ_Mexico_Framework.pdf'

In [14]:
import graphviz

# =====================================================
# CEQ INCOME CONCEPTS - MEXICO
# EXTENDED VERSION WITH DATA GENERATION PROCESS
# =====================================================

dot = graphviz.Digraph(comment='CEQ Fiscal Incidence Diagram')

# -----------------------------------------------------
# GLOBAL LAYOUT
# -----------------------------------------------------

dot.attr(
    rankdir='TB',
    splines='ortho',
    nodesep='0.9',
    ranksep='1.4',
    fontname='Times New Roman',
    fontsize='12',
    pad='0.3',
    ratio='compress'
)

# -----------------------------------------------------
# COLOR PALETTE
# -----------------------------------------------------

color_income = '#1F618D'
color_tax = '#A93226'
color_transfer = '#2E86C1'
color_subsidy = '#17A589'
color_process = '#7D3C98'
color_data = '#566573'
color_box = '#F4F6F9'

# -----------------------------------------------------
# DATA SOURCE
# -----------------------------------------------------

dot.node(
    'ENIGH',
    'ENIGH household survey data\nIncome reported after tax withholding',
    shape='box',
    style='rounded,filled',
    fillcolor=color_data,
    fontcolor='white'
)

# -----------------------------------------------------
# NET MARKET INCOME (OBSERVED)
# -----------------------------------------------------

dot.node(
    'NMI',
    'Net Market Income (In)\nObserved income after ISR withholding',
    shape='ellipse',
    style='rounded,filled',
    fillcolor=color_income,
    fontcolor='white',
    width='3'
)

# -----------------------------------------------------
# GROSS-UP PROCEDURE
# -----------------------------------------------------

dot.node(
    'GrossUp',
    'Gross-up procedure\nAnalytical inversion of ISR schedule',
    shape='box',
    style='rounded,filled',
    fillcolor=color_process,
    fontcolor='white',
    width='3'
)

# -----------------------------------------------------
# MARKET INCOME
# -----------------------------------------------------

dot.node(
    'MI',
    'Market Income (Im)\nRecovered gross income before taxes',
    shape='ellipse',
    style='rounded,filled',
    fillcolor=color_income,
    fontcolor='white',
    width='3'
)

# -----------------------------------------------------
# DIRECT TRANSFERS
# -----------------------------------------------------

dot.node(
    'DirectTrans',
    'Direct Transfers\nCash programs',
    shape='box',
    style='rounded,filled',
    fillcolor=color_transfer,
    fontcolor='white'
)

# -----------------------------------------------------
# DISPOSABLE INCOME
# -----------------------------------------------------

dot.node(
    'DI',
    'Disposable Income (Id)\nAfter direct transfers',
    shape='ellipse',
    style='rounded,filled',
    fillcolor=color_income,
    fontcolor='white',
    width='3'
)

# -----------------------------------------------------
# CONSUMPTION
# -----------------------------------------------------

dot.node(
    'Consumption',
    'Household consumption expenditure\nDerived from ENIGH spending records',
    shape='box',
    style='rounded,filled',
    fillcolor=color_box
)

# -----------------------------------------------------
# INDIRECT TAXES
# -----------------------------------------------------

dot.node(
    'IVA',
    'Value Added Tax (IVA)\nEstimated from taxable formal purchases',
    shape='box',
    style='rounded,filled',
    fillcolor=color_tax,
    fontcolor='white'
)

dot.node(
    'Electricity',
    'Electricity payments\nIndirect tax component',
    shape='box',
    style='rounded,filled',
    fillcolor=color_tax,
    fontcolor='white'
)

# -----------------------------------------------------
# POST-FISCAL INCOME
# -----------------------------------------------------

dot.node(
    'PFI',
    'Post-fiscal Income (Ipf)\nAfter indirect taxes',
    shape='ellipse',
    style='rounded,filled',
    fillcolor=color_income,
    fontcolor='white',
    width='3'
)

# -----------------------------------------------------
# IN-KIND TRANSFERS
# -----------------------------------------------------

dot.node(
    'InKind',
    'In-kind transfers\nHealth and education services',
    shape='box',
    style='rounded,filled',
    fillcolor=color_transfer,
    fontcolor='white'
)

# -----------------------------------------------------
# FINAL INCOME
# -----------------------------------------------------

dot.node(
    'FI',
    'Final Income (If)',
    shape='ellipse',
    style='rounded,filled',
    fillcolor=color_income,
    fontcolor='white',
    width='3'
)

# -----------------------------------------------------
# RESULTS
# -----------------------------------------------------

dot.node(
    'Results',
    'Redistributive impact indicators\nGini coefficient\nPoverty headcount\nTax burden by decile',
    shape='box',
    style='rounded,filled',
    fillcolor=color_box
)

# -----------------------------------------------------
# MAIN FLOW
# -----------------------------------------------------

dot.edge('ENIGH', 'NMI')

dot.edge('NMI', 'GrossUp')

dot.edge('GrossUp', 'MI')

dot.edge('MI', 'DI')

dot.edge('DirectTrans', 'DI')

dot.edge('DI', 'PFI')

dot.edge('Consumption', 'IVA')

dot.edge('Consumption', 'Electricity')

dot.edge('IVA', 'PFI')

dot.edge('Electricity', 'PFI')

dot.edge('PFI', 'FI')

dot.edge('InKind', 'FI')

dot.edge('FI', 'Results')

# -----------------------------------------------------
# FOOTNOTE
# -----------------------------------------------------

dot.node(
    'Note',
    'Note: ENIGH income is reported after tax withholding.\nMarket Income is recovered analytically using the statutory ISR schedule.',
    shape='plaintext',
    fontsize='10'
)

dot.edge(
    'Note',
    'NMI',
    style='dashed',
    color='gray'
)

# -----------------------------------------------------
# RENDER
# -----------------------------------------------------

dot.render(
    'CEQ_Mexico_extended_pipeline',
    format='pdf',
    cleanup=True
)

'CEQ_Mexico_extended_pipeline.pdf'

# LAPTOP WINDOWS 

In [ ]:
import graphviz

dot = graphviz.Digraph(comment='CEQ Fiscal Incidence Pipeline')

# -----------------------------------------------------
# CONFIGURACIÓN GLOBAL (SOLO FORMATO)
# -----------------------------------------------------
dot.attr(
    rankdir='TB',
    splines='ortho',

    size='8.5,11',
    ratio='fill',

    nodesep='0.7',
    ranksep='0.75',

    fontname='Helvetica',
    fontsize='16',
    pad='0.4'
)

# -----------------------------------------------------
# PALETA (SIN CAMBIOS)
# -----------------------------------------------------
color_income = '#1F4E79'
color_add = '#4F81BD'
color_sub = '#7F8C8D'
color_def = '#5D6D7E'
color_source = '#A6ACAF'

# -----------------------------------------------------
# DEFINICIONES (MISMO TEXTO)
# -----------------------------------------------------
with dot.subgraph() as s:
    s.attr(rank='same')

    s.node(
        'def_MI',
        'Salarios, rentas,\nautoempleo y pensiones.',
        shape='none',
        fontcolor=color_def,
        fontsize='13'
    )

    s.node(
        'def_NMI',
        'Ingreso tras pagar\nimpuestos directos.',
        shape='none',
        fontcolor=color_def,
        fontsize='13'
    )

    s.node(
        'def_DI',
        'Efectivo disponible\nincluyendo transferencias.',
        shape='none',
        fontcolor=color_def,
        fontsize='13'
    )

    s.node(
        'def_PFI',
        'Poder de compra neto\ntras impuestos al consumo.',
        shape='none',
        fontcolor=color_def,
        fontsize='13'
    )

    s.node(
        'def_FI',
        'Bienestar total: incluye\nvalor de salud y educación.',
        shape='none',
        fontcolor=color_def,
        fontsize='13'
    )

# -----------------------------------------------------
# CONCEPTOS DE INGRESO (MISMA INFORMACIÓN)
# -----------------------------------------------------
dot.node(
    'MI',
    'Market Income\n(Im)',
    shape='ellipse',
    style='rounded,filled',
    fillcolor=color_income,
    fontcolor='white',

    width='3.8',
    height='1.3',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'NMI',
    'Net Market Income\n(In)',
    shape='ellipse',
    style='rounded,filled',
    fillcolor=color_income,
    fontcolor='white',

    width='3.8',
    height='1.3',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'DI',
    'Disposable Income\n(Id)',
    shape='ellipse',
    style='rounded,filled',
    fillcolor=color_income,
    fontcolor='white',

    width='3.8',
    height='1.3',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'PFI',
    'Post-fiscal Income\n(Ipf)',
    shape='ellipse',
    style='rounded,filled',
    fillcolor=color_income,
    fontcolor='white',

    width='3.8',
    height='1.3',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'FI',
    'Final Income\n(If)',
    shape='ellipse',
    style='rounded,filled',
    fillcolor=color_income,
    fontcolor='white',

    width='3.8',
    height='1.3',

    fontsize='16',
    penwidth='1.3'
)

# -----------------------------------------------------
# OPERACIONES (MISMO TEXTO)
# -----------------------------------------------------
dot.node(
    'DT',
    '(-) Impuestos Directos ISR',
    shape='box',
    style='rounded,filled',
    fillcolor=color_sub,
    fontcolor='white',

    width='3.8',
    height='1.2',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'GT',
    '(+) Transferencias\nDirectas \nGobierno',
    shape='box',
    style='rounded,filled',
    fillcolor=color_add,
    fontcolor='white',

    width='3.8',
    height='1.2',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'IS',
    '(+) Subsidios Indirectos \n(Subsidio a la electricidad)',
    shape='box',
    style='rounded,filled',
    fillcolor=color_add,
    fontcolor='white',

    width='3.8',
    height='1.2',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'IT',
    '(-) Impuestos Indirectos\n(IVA / IEPS)',
    shape='box',
    style='rounded,filled',
    fillcolor=color_sub,
    fontcolor='white',

    width='3.8',
    height='1.2',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'IK',
    '(+) Educación y Salud\n(Transferencias en especie)',
    shape='box',
    style='rounded,filled',
    fillcolor=color_add,
    fontcolor='white',

    width='3.8',
    height='1.2',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'CP',
    '(-) Copagos',
    shape='box',
    style='rounded,filled',
    fillcolor=color_sub,
    fontcolor='white',

    width='3.8',
    height='1.2',

    fontsize='16',
    penwidth='1.3'
)

# -----------------------------------------------------
# CONEXIONES (SIN CAMBIOS)
# -----------------------------------------------------

dot.edge('def_MI', 'MI', style='dotted', arrowhead='none')
dot.edge('def_NMI', 'NMI', style='dotted', arrowhead='none')
dot.edge('def_DI', 'DI', style='dotted', arrowhead='none')
dot.edge('def_PFI', 'PFI', style='dotted', arrowhead='none')
dot.edge('def_FI', 'FI', style='dotted', arrowhead='none')

dot.edge('MI', 'DT', style='solid')
dot.edge('DT', 'NMI', style='solid')
dot.edge('NMI', 'GT', style='solid')
dot.edge('GT', 'DI', style='solid')
dot.edge('DI', 'IS', style='solid')
dot.edge('IS', 'IT', style='solid')
dot.edge('IT', 'PFI', style='solid')
dot.edge('PFI', 'IK', style='solid')
dot.edge('IK', 'CP', style='solid')
dot.edge('CP', 'FI', style='solid')

dot.edge('NMI', 'Source. ENIGH', style='dashed', arrowhead='none')
dot.edge('DT', 'Source. SAT', style='dashed', arrowhead='none')
dot.edge('GT', 'Source. To be defined', style='dashed', arrowhead='none')
dot.edge('IT', 'Source. To be defined', style='dashed', arrowhead='none')
dot.edge('IK', 'Source. To be defined', style='dashed', arrowhead='none')

# -----------------------------------------------------
# EXPORTACIÓN
# -----------------------------------------------------
dot.render(
    'Metodologia_CEQ_Formato_Academico',
    format='pdf',
    cleanup=True
)

print("Diagrama generado con cuadros grandes.")

Diagrama generado con cuadros grandes.


In [17]:
import graphviz

dot = graphviz.Digraph(comment='CEQ Fiscal Incidence Pipeline')

# -----------------------------------------------------
# GLOBAL CONFIGURATION (FORMAT ONLY)
# -----------------------------------------------------
dot.attr(
    rankdir='TB',
    splines='ortho',

    size='8.5,11',
    ratio='fill',

    nodesep='0.7',
    ranksep='0.75',

    fontname='Helvetica',
    fontsize='16',
    pad='0.4'
)

# -----------------------------------------------------
# PALETTE (NO CHANGES)
# -----------------------------------------------------
color_income = '#1F4E79'
color_add = '#4F81BD'
color_sub = '#7F8C8D'
color_def = '#5D6D7E'
color_source = '#A6ACAF'

# -----------------------------------------------------
# DEFINITIONS (SAME TEXT)
# -----------------------------------------------------
with dot.subgraph() as s:
    s.attr(rank='same')

    s.node(
        'def_MI',
        'Wages, rents,\nself-employment and pensions.',
        shape='none',
        fontcolor=color_def,
        fontsize='13'
    )

    s.node(
        'def_NMI',
        'Income after paying\ndirect taxes.',
        shape='none',
        fontcolor=color_def,
        fontsize='13'
    )

    s.node(
        'def_DI',
        'Available cash\nincluding transfers.',
        shape='none',
        fontcolor=color_def,
        fontsize='13'
    )

    s.node(
        'def_PFI',
        'Net purchasing power\nafter consumption taxes.',
        shape='none',
        fontcolor=color_def,
        fontsize='13'
    )

    s.node(
        'def_FI',
        'Total welfare: includes\nvalue of health and education.',
        shape='none',
        fontcolor=color_def,
        fontsize='13'
    )

# -----------------------------------------------------
# INCOME CONCEPTS (SAME INFORMATION)
# -----------------------------------------------------
dot.node(
    'MI',
    'Market Income\n(Im)',
    shape='ellipse',
    style='rounded,filled',
    fillcolor=color_income,
    fontcolor='white',

    width='3.8',
    height='1.3',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'NMI',
    'Net Market Income\n(In)',
    shape='ellipse',
    style='rounded,filled',
    fillcolor=color_income,
    fontcolor='white',

    width='3.8',
    height='1.3',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'DI',
    'Disposable Income\n(Id)',
    shape='ellipse',
    style='rounded,filled',
    fillcolor=color_income,
    fontcolor='white',

    width='3.8',
    height='1.3',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'PFI',
    'Post-fiscal Income\n(Ipf)',
    shape='ellipse',
    style='rounded,filled',
    fillcolor=color_income,
    fontcolor='white',

    width='3.8',
    height='1.3',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'FI',
    'Final Income\n(If)',
    shape='ellipse',
    style='rounded,filled',
    fillcolor=color_income,
    fontcolor='white',

    width='3.8',
    height='1.3',

    fontsize='16',
    penwidth='1.3'
)

# -----------------------------------------------------
# OPERATIONS (SAME TEXT)
# -----------------------------------------------------
dot.node(
    'DT',
    '(-) Direct Taxes ISR',
    shape='box',
    style='rounded,filled',
    fillcolor=color_sub,
    fontcolor='white',

    width='3.8',
    height='1.2',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'GT',
    '(+) Direct Government\nTransfers',
    shape='box',
    style='rounded,filled',
    fillcolor=color_add,
    fontcolor='white',

    width='3.8',
    height='1.2',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'IS',
    '(+) Indirect Subsidies\n(Electricity subsidy)',
    shape='box',
    style='rounded,filled',
    fillcolor=color_add,
    fontcolor='white',

    width='3.8',
    height='1.2',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'IT',
    '(-) Indirect Taxes\n(VAT / IEPS)',
    shape='box',
    style='rounded,filled',
    fillcolor=color_sub,
    fontcolor='white',

    width='3.8',
    height='1.2',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'IK',
    '(+) Education and Health\n(In-kind transfers)',
    shape='box',
    style='rounded,filled',
    fillcolor=color_add,
    fontcolor='white',

    width='3.8',
    height='1.2',

    fontsize='16',
    penwidth='1.3'
)

dot.node(
    'CP',
    '(-) Copayments',
    shape='box',
    style='rounded,filled',
    fillcolor=color_sub,
    fontcolor='white',

    width='3.8',
    height='1.2',

    fontsize='16',
    penwidth='1.3'
)

# -----------------------------------------------------
# CONNECTIONS (NO CHANGES)
# -----------------------------------------------------

dot.edge('def_MI', 'MI', style='dotted', arrowhead='none')
dot.edge('def_NMI', 'NMI', style='dotted', arrowhead='none')
dot.edge('def_DI', 'DI', style='dotted', arrowhead='none')
dot.edge('def_PFI', 'PFI', style='dotted', arrowhead='none')
dot.edge('def_FI', 'FI', style='dotted', arrowhead='none')

dot.edge('MI', 'DT', style='solid')
dot.edge('DT', 'NMI', style='solid')
dot.edge('NMI', 'GT', style='solid')
dot.edge('GT', 'DI', style='solid')
dot.edge('DI', 'IS', style='solid')
dot.edge('IS', 'IT', style='solid')
dot.edge('IT', 'PFI', style='solid')
dot.edge('PFI', 'IK', style='solid')
dot.edge('IK', 'CP', style='solid')
dot.edge('CP', 'FI', style='solid')

dot.edge('NMI', 'Source. ENIGH', style='dashed', arrowhead='none')
dot.edge('DT', 'Source. SAT', style='dashed', arrowhead='none')
dot.edge('GT', 'Source. To define', style='dashed', arrowhead='none')
dot.edge('IT', 'Source. To define', style='dashed', arrowhead='none')
dot.edge('IK', 'Source. To define', style='dashed', arrowhead='none')

# -----------------------------------------------------
# EXPORT
# -----------------------------------------------------
dot.render(
    'Metodologia_CEQ_Formato_Academico',
    format='pdf',
    cleanup=True
)

print("Diagram generated with large boxes.")

Diagram generated with large boxes.


# modificacion 2

In [18]:
import graphviz

dot = graphviz.Digraph(comment='CEQ Fiscal Incidence Pipeline')

# -----------------------------------------------------
# CONFIGURACIÓN GLOBAL (SIN CAMBIOS)
# -----------------------------------------------------
dot.attr(
    rankdir='TB',
    splines='ortho',
    size='8.5,11',
    ratio='fill',
    nodesep='0.7',
    ranksep='0.75',
    fontname='Helvetica',
    fontsize='16',
    pad='0.4'
)

# -----------------------------------------------------
# PALETA (SIN CAMBIOS)
# -----------------------------------------------------
color_income = '#1F4E79'
color_add = '#4F81BD'
color_sub = '#7F8C8D'
color_def = '#5D6D7E'
color_source = '#A6ACAF'

# =====================================================
# NUEVO: CLUSTER IZQUIERDO (Definiciones + Fuentes)
# =====================================================
with dot.subgraph(name='cluster_left') as left:
    left.attr(label='Definiciones y fuentes de datos', style='rounded', color=color_source, fontcolor=color_source)
    left.attr(rank='same')  # para que estén todas a la izquierda, aunque se ordenarán verticalmente

    # Definiciones (ya existían, las movemos dentro del cluster)
    left.node('def_MI', 'Salarios, rentas,\nautoempleo y pensiones.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_NMI', 'Ingreso tras pagar\nimpuestos directos.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_DI', 'Efectivo disponible\nincluyendo transferencias.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_PFI', 'Poder de compra neto\ntras impuestos al consumo.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_FI', 'Bienestar total: incluye\nvalor de salud y educación.', shape='none', fontcolor=color_def, fontsize='13')

    # NUEVO: Crear explícitamente los nodos de fuentes (antes eran implícitos)
    left.node('Source_ENIGH', 'Source. ENIGH', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='12')
    left.node('Source_SAT', 'Source. SAT', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='12')
    left.node('Source_ToDefine', 'Source. To define', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='12')

    # Orden vertical: definiciones arriba, fuentes abajo (usando aristas invisibles)
    left.edge('def_MI', 'def_NMI', style='invis')
    left.edge('def_NMI', 'def_DI', style='invis')
    left.edge('def_DI', 'def_PFI', style='invis')
    left.edge('def_PFI', 'def_FI', style='invis')
    left.edge('def_FI', 'Source_ENIGH', style='invis')
    left.edge('Source_ENIGH', 'Source_SAT', style='invis')
    left.edge('Source_SAT', 'Source_ToDefine', style='invis')

# =====================================================
# CLUSTER CENTRAL (Escalera de ingresos) - SIN CAMBIOS EN NODOS
# =====================================================
with dot.subgraph(name='cluster_center') as center:
    center.attr(label='Conceptos de ingreso', style='rounded', color=color_income, fontcolor=color_income)
    center.attr(rank='same')  # se ordenarán verticalmente con aristas

    center.node('MI', 'Market Income\n(Im)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('NMI', 'Net Market Income\n(In)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('DI', 'Disposable Income\n(Id)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('PFI', 'Post-fiscal Income\n(Ipf)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('FI', 'Final Income\n(If)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')

    # Forzar orden vertical (la escalera)
    center.edge('MI', 'NMI', style='invis')
    center.edge('NMI', 'DI', style='invis')
    center.edge('DI', 'PFI', style='invis')
    center.edge('PFI', 'FI', style='invis')

# =====================================================
# CLUSTER DERECHO (Operaciones: sumas y restas)
# =====================================================
with dot.subgraph(name='cluster_right') as right:
    right.attr(label='Modificaciones al ingreso', style='rounded', color=color_add, fontcolor=color_add)
    right.attr(rank='same')

    right.node('DT', '(-) Impuestos Directos ISR', shape='box', style='rounded,filled', fillcolor=color_sub, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('GT', '(+) Transferencias\nDirectas \nGobierno', shape='box', style='rounded,filled', fillcolor=color_add, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('IS', '(+) Subsidios Indirectos \n(Subsidio a la electricidad)', shape='box', style='rounded,filled', fillcolor=color_add, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('IT', '(-) Impuestos Indirectos\n(IVA / IEPS)', shape='box', style='rounded,filled', fillcolor=color_sub, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('IK', '(+) Educación y Salud\n(Transferencias en especie)', shape='box', style='rounded,filled', fillcolor=color_add, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('CP', '(-) Copagos', shape='box', style='rounded,filled', fillcolor=color_sub, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')

    # Orden vertical (paralelo a la escalera)
    right.edge('DT', 'GT', style='invis')
    right.edge('GT', 'IS', style='invis')
    right.edge('IS', 'IT', style='invis')
    right.edge('IT', 'IK', style='invis')
    right.edge('IK', 'CP', style='invis')

# =====================================================
# CONEXIONES ORIGINALES (TODAS SE MANTIENEN IGUALES)
# =====================================================
# Definiciones a conceptos de ingreso (líneas punteadas)
dot.edge('def_MI', 'MI', style='dotted', arrowhead='none')
dot.edge('def_NMI', 'NMI', style='dotted', arrowhead='none')
dot.edge('def_DI', 'DI', style='dotted', arrowhead='none')
dot.edge('def_PFI', 'PFI', style='dotted', arrowhead='none')
dot.edge('def_FI', 'FI', style='dotted', arrowhead='none')

# Flechas de la escalera (operaciones entre ingresos)
dot.edge('MI', 'DT', style='solid')
dot.edge('DT', 'NMI', style='solid')
dot.edge('NMI', 'GT', style='solid')
dot.edge('GT', 'DI', style='solid')
dot.edge('DI', 'IS', style='solid')
dot.edge('IS', 'IT', style='solid')
dot.edge('IT', 'PFI', style='solid')
dot.edge('PFI', 'IK', style='solid')
dot.edge('IK', 'CP', style='solid')
dot.edge('CP', 'FI', style='solid')

# Fuentes de datos (usando los nodos explícitos creados)
dot.edge('NMI', 'Source_ENIGH', style='dashed', arrowhead='none')
dot.edge('DT', 'Source_SAT', style='dashed', arrowhead='none')
dot.edge('GT', 'Source_ToDefine', style='dashed', arrowhead='none')
dot.edge('IT', 'Source_ToDefine', style='dashed', arrowhead='none')
dot.edge('IK', 'Source_ToDefine', style='dashed', arrowhead='none')

# =====================================================
# EXPORTACIÓN
# =====================================================
dot.render('Metodologia_CEQ_Formato_Academico', format='pdf', cleanup=True)
print("Diagrama generado con cuadrantes (izquierda: definiciones+fuentes, centro: ingresos, derecha: operaciones).")

Diagrama generado con cuadrantes (izquierda: definiciones+fuentes, centro: ingresos, derecha: operaciones).


# modificaciones 3

In [23]:
import graphviz

dot = graphviz.Digraph(comment='CEQ Fiscal Incidence Pipeline')

# -----------------------------------------------------
# CONFIGURACIÓN GLOBAL (SIN newrank PARA EVITAR ERRORES)
# -----------------------------------------------------
dot.attr(
    rankdir='TB',
    splines='ortho',
    size='8.5,11',
    ratio='fill',
    nodesep='0.7',
    ranksep='0.75',
    fontname='Helvetica',
    fontsize='16',
    pad='0.4'
)

# -----------------------------------------------------
# PALETA
# -----------------------------------------------------
color_income = '#1F4E79'
color_add = '#4F81BD'
color_sub = '#7F8C8D'
color_def = '#5D6D7E'
color_source = '#A6ACAF'

# =====================================================
# CLUSTER IZQUIERDO (DEFINICIONES)
# =====================================================
with dot.subgraph(name='cluster_left') as left:
    left.attr(label='Definiciones', style='rounded', color=color_def, fontcolor=color_def)
    left.node('def_MI', 'Salarios, rentas,\nautoempleo y pensiones.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_NMI', 'Ingreso tras pagar\nimpuestos directos.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_DI', 'Efectivo disponible\nincluyendo transferencias.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_PFI', 'Poder de compra neto\ntras impuestos al consumo.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_FI', 'Bienestar total: incluye\nvalor de salud y educación.', shape='none', fontcolor=color_def, fontsize='13')
    # orden vertical con invisibles
    left.edge('def_MI', 'def_NMI', style='invis')
    left.edge('def_NMI', 'def_DI', style='invis')
    left.edge('def_DI', 'def_PFI', style='invis')
    left.edge('def_PFI', 'def_FI', style='invis')

# =====================================================
# CLUSTER CENTRAL (ESCALERA DE INGRESOS)
# =====================================================
with dot.subgraph(name='cluster_center') as center:
    center.attr(label='Conceptos de ingreso', style='rounded', color=color_income, fontcolor=color_income)
    center.node('MI', 'Market Income\n(Im)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('NMI', 'Net Market Income\n(In)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('DI', 'Disposable Income\n(Id)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('PFI', 'Post-fiscal Income\n(Ipf)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('FI', 'Final Income\n(If)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    # escalera vertical
    center.edge('MI', 'NMI', style='invis')
    center.edge('NMI', 'DI', style='invis')
    center.edge('DI', 'PFI', style='invis')
    center.edge('PFI', 'FI', style='invis')

# =====================================================
# CLUSTER DERECHO (OPERACIONES)
# =====================================================
with dot.subgraph(name='cluster_right') as right:
    right.attr(label='Modificaciones al ingreso', style='rounded', color=color_add, fontcolor=color_add)
    right.node('DT', '(-) Impuestos Directos ISR', shape='box', style='rounded,filled', fillcolor=color_sub, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('GT', '(+) Transferencias\nDirectas \nGobierno', shape='box', style='rounded,filled', fillcolor=color_add, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('IS', '(+) Subsidios Indirectos \n(Subsidio a la electricidad)', shape='box', style='rounded,filled', fillcolor=color_add, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('IT', '(-) Impuestos Indirectos\n(IVA / IEPS)', shape='box', style='rounded,filled', fillcolor=color_sub, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('IK', '(+) Educación y Salud\n(Transferencias en especie)', shape='box', style='rounded,filled', fillcolor=color_add, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('CP', '(-) Copagos', shape='box', style='rounded,filled', fillcolor=color_sub, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    # orden vertical
    right.edge('DT', 'GT', style='invis')
    right.edge('GT', 'IS', style='invis')
    right.edge('IS', 'IT', style='invis')
    right.edge('IT', 'IK', style='invis')
    right.edge('IK', 'CP', style='invis')

# =====================================================
# FUENTES DE DATOS: DEBAJO DE CADA RECUADRO (SIN LÍNEAS LARGAS)
# Usamos aristas invisibles para forzar posición vertical y luego una punteada muy corta
# =====================================================

# Nodo fuente para NMI
dot.node('src_NMI', 'Source. ENIGH', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='11', margin='0.2')
dot.edge('NMI', 'src_NMI', style='invis')  # fuerza que src_NMI esté debajo de NMI
dot.edge('NMI', 'src_NMI', style='dotted', arrowhead='none', penwidth='0.8')  # línea corta

# Fuente para DT
dot.node('src_DT', 'Source. SAT', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='11', margin='0.2')
dot.edge('DT', 'src_DT', style='invis')
dot.edge('DT', 'src_DT', style='dotted', arrowhead='none', penwidth='0.8')

# Fuente para GT
dot.node('src_GT', 'Source. To define', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='11', margin='0.2')
dot.edge('GT', 'src_GT', style='invis')
dot.edge('GT', 'src_GT', style='dotted', arrowhead='none', penwidth='0.8')

# Fuente para IT
dot.node('src_IT', 'Source. To define', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='11', margin='0.2')
dot.edge('IT', 'src_IT', style='invis')
dot.edge('IT', 'src_IT', style='dotted', arrowhead='none', penwidth='0.8')

# Fuente para IK
dot.node('src_IK', 'Source. To define', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='11', margin='0.2')
dot.edge('IK', 'src_IK', style='invis')
dot.edge('IK', 'src_IK', style='dotted', arrowhead='none', penwidth='0.8')

# =====================================================
# CONEXIONES ORIGINALES (DEFINICIONES A INGRESOS Y ESCALERA)
# =====================================================
dot.edge('def_MI', 'MI', style='dotted', arrowhead='none')
dot.edge('def_NMI', 'NMI', style='dotted', arrowhead='none')
dot.edge('def_DI', 'DI', style='dotted', arrowhead='none')
dot.edge('def_PFI', 'PFI', style='dotted', arrowhead='none')
dot.edge('def_FI', 'FI', style='dotted', arrowhead='none')

dot.edge('MI', 'DT', style='solid')
dot.edge('DT', 'NMI', style='solid')
dot.edge('NMI', 'GT', style='solid')
dot.edge('GT', 'DI', style='solid')
dot.edge('DI', 'IS', style='solid')
dot.edge('IS', 'IT', style='solid')
dot.edge('IT', 'PFI', style='solid')
dot.edge('PFI', 'IK', style='solid')
dot.edge('IK', 'CP', style='solid')
dot.edge('CP', 'FI', style='solid')

# =====================================================
# EXPORTACIÓN
# =====================================================
dot.render('Metodologia_CEQ_Formato_Academico', format='pdf', cleanup=True)
print("Diagrama generado con fuentes debajo de cada recuadro, líneas cortas y sin errores.")

Error: Could not open "Metodologia_CEQ_Formato_Academico.pdf" for writing : Permission denied


CalledProcessError: Command '[WindowsPath('dot'), '-Kdot', '-Tpdf', '-O', 'Metodologia_CEQ_Formato_Academico']' returned non-zero exit status 1. [stderr: b'Error: Could not open "Metodologia_CEQ_Formato_Academico.pdf" for writing : Permission denied\r\n']

In [24]:
import graphviz

dot = graphviz.Digraph(comment='CEQ Fiscal Incidence Pipeline')

# -----------------------------------------------------
# CONFIGURACIÓN GLOBAL
# -----------------------------------------------------
dot.attr(
    rankdir='TB',
    splines='ortho',
    size='8.5,11',
    ratio='fill',
    nodesep='0.7',
    ranksep='0.75',
    fontname='Helvetica',
    fontsize='16',
    pad='0.4'
)

# -----------------------------------------------------
# PALETA
# -----------------------------------------------------
color_income = '#1F4E79'
color_add = '#4F81BD'
color_sub = '#7F8C8D'
color_def = '#5D6D7E'
color_source = '#A6ACAF'

# =====================================================
# CLUSTER IZQUIERDO (DEFINICIONES)
# =====================================================
with dot.subgraph(name='cluster_left') as left:
    left.attr(label='Definiciones', style='rounded', color=color_def, fontcolor=color_def)
    left.node('def_MI', 'Salarios, rentas,\nautoempleo y pensiones.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_NMI', 'Ingreso tras pagar\nimpuestos directos.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_DI', 'Efectivo disponible\nincluyendo transferencias.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_PFI', 'Poder de compra neto\ntras impuestos al consumo.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_FI', 'Bienestar total: incluye\nvalor de salud y educación.', shape='none', fontcolor=color_def, fontsize='13')
    # orden vertical con invisibles
    left.edge('def_MI', 'def_NMI', style='invis')
    left.edge('def_NMI', 'def_DI', style='invis')
    left.edge('def_DI', 'def_PFI', style='invis')
    left.edge('def_PFI', 'def_FI', style='invis')

# =====================================================
# CLUSTER CENTRAL (ESCALERA DE INGRESOS)
# =====================================================
with dot.subgraph(name='cluster_center') as center:
    center.attr(label='Conceptos de ingreso', style='rounded', color=color_income, fontcolor=color_income)
    center.node('MI', 'Market Income\n(Im)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('NMI', 'Net Market Income\n(In)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('DI', 'Disposable Income\n(Id)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('PFI', 'Post-fiscal Income\n(Ipf)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('FI', 'Final Income\n(If)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    # escalera vertical
    center.edge('MI', 'NMI', style='invis')
    center.edge('NMI', 'DI', style='invis')
    center.edge('DI', 'PFI', style='invis')
    center.edge('PFI', 'FI', style='invis')

# =====================================================
# CLUSTER DERECHO (OPERACIONES)
# =====================================================
with dot.subgraph(name='cluster_right') as right:
    right.attr(label='Modificaciones al ingreso', style='rounded', color=color_add, fontcolor=color_add)
    right.node('DT', '(-) Impuestos Directos ISR', shape='box', style='rounded,filled', fillcolor=color_sub, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('GT', '(+) Transferencias\nDirectas \nGobierno', shape='box', style='rounded,filled', fillcolor=color_add, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('IS', '(+) Subsidios Indirectos \n(Subsidio a la electricidad)', shape='box', style='rounded,filled', fillcolor=color_add, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('IT', '(-) Impuestos Indirectos\n(IVA / IEPS)', shape='box', style='rounded,filled', fillcolor=color_sub, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('IK', '(+) Educación y Salud\n(Transferencias en especie)', shape='box', style='rounded,filled', fillcolor=color_add, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('CP', '(-) Copagos', shape='box', style='rounded,filled', fillcolor=color_sub, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    # orden vertical
    right.edge('DT', 'GT', style='invis')
    right.edge('GT', 'IS', style='invis')
    right.edge('IS', 'IT', style='invis')
    right.edge('IT', 'IK', style='invis')
    right.edge('IK', 'CP', style='invis')

# =====================================================
# FUENTES DE DATOS (DEBAJO DE CADA RECUADRO, LÍNEA CORTA)
# =====================================================
# Fuente para NMI
dot.node('src_NMI', 'Source. ENIGH', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='11', margin='0.2')
dot.edge('NMI', 'src_NMI', style='invis')
dot.edge('NMI', 'src_NMI', style='dotted', arrowhead='none', penwidth='0.8')

# Fuente para DT
dot.node('src_DT', 'Source. SAT', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='11', margin='0.2')
dot.edge('DT', 'src_DT', style='invis')
dot.edge('DT', 'src_DT', style='dotted', arrowhead='none', penwidth='0.8')

# Fuente para GT
dot.node('src_GT', 'Source. To define', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='11', margin='0.2')
dot.edge('GT', 'src_GT', style='invis')
dot.edge('GT', 'src_GT', style='dotted', arrowhead='none', penwidth='0.8')

# Fuente para IT
dot.node('src_IT', 'Source. To define', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='11', margin='0.2')
dot.edge('IT', 'src_IT', style='invis')
dot.edge('IT', 'src_IT', style='dotted', arrowhead='none', penwidth='0.8')

# Fuente para IK
dot.node('src_IK', 'Source. To define', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='11', margin='0.2')
dot.edge('IK', 'src_IK', style='invis')
dot.edge('IK', 'src_IK', style='dotted', arrowhead='none', penwidth='0.8')

# =====================================================
# CONEXIONES ORIGINALES
# =====================================================
dot.edge('def_MI', 'MI', style='dotted', arrowhead='none')
dot.edge('def_NMI', 'NMI', style='dotted', arrowhead='none')
dot.edge('def_DI', 'DI', style='dotted', arrowhead='none')
dot.edge('def_PFI', 'PFI', style='dotted', arrowhead='none')
dot.edge('def_FI', 'FI', style='dotted', arrowhead='none')

dot.edge('MI', 'DT', style='solid')
dot.edge('DT', 'NMI', style='solid')
dot.edge('NMI', 'GT', style='solid')
dot.edge('GT', 'DI', style='solid')
dot.edge('DI', 'IS', style='solid')
dot.edge('IS', 'IT', style='solid')
dot.edge('IT', 'PFI', style='solid')
dot.edge('PFI', 'IK', style='solid')
dot.edge('IK', 'CP', style='solid')
dot.edge('CP', 'FI', style='solid')

# =====================================================
# EXPORTACIÓN: PDF Y PNG (para insertar en Word)
# =====================================================
dot.render('Metodologia_CEQ_Formato_Academico', format='pdf', cleanup=True)
dot.render('Metodologia_CEQ_Formato_Academico', format='png', cleanup=True)

print("✅ Diagrama generado en PDF y PNG. El archivo PNG puedes insertarlo directamente en Word.")

✅ Diagrama generado en PDF y PNG. El archivo PNG puedes insertarlo directamente en Word.


In [26]:
import graphviz

dot = graphviz.Digraph(comment='CEQ Fiscal Incidence Pipeline')

# -----------------------------------------------------
# GLOBAL CONFIGURATION
# -----------------------------------------------------
dot.attr(
    rankdir='TB',
    splines='ortho',
    size='8.5,11',
    ratio='fill',
    nodesep='0.7',
    ranksep='0.75',
    fontname='Helvetica',
    fontsize='16',
    pad='0.4'
)

# -----------------------------------------------------
# PALETTE
# -----------------------------------------------------
color_income = '#1F4E79'
color_add = '#4F81BD'
color_sub = '#7F8C8D'
color_def = '#5D6D7E'
color_source = '#A6ACAF'

# =====================================================
# LEFT CLUSTER (DEFINITIONS)
# =====================================================
with dot.subgraph(name='cluster_left') as left:
    left.attr(label='Definitions', style='rounded', color=color_def, fontcolor=color_def)
    left.node('def_MI', 'Wages, rents,\nself-employment and pensions.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_NMI', 'Income after paying\ndirect taxes.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_DI', 'Available cash\nincluding transfers.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_PFI', 'Net purchasing power\nafter consumption taxes.', shape='none', fontcolor=color_def, fontsize='13')
    left.node('def_FI', 'Total welfare: includes\nvalue of health and education.', shape='none', fontcolor=color_def, fontsize='13')
    # vertical order with invisible edges
    left.edge('def_MI', 'def_NMI', style='invis')
    left.edge('def_NMI', 'def_DI', style='invis')
    left.edge('def_DI', 'def_PFI', style='invis')
    left.edge('def_PFI', 'def_FI', style='invis')

# =====================================================
# CENTRAL CLUSTER (INCOME CONCEPTS)
# =====================================================
with dot.subgraph(name='cluster_center') as center:
    center.attr(label='Income Concepts', style='rounded', color=color_income, fontcolor=color_income)
    center.node('MI', 'Market Income\n(Im)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('NMI', 'Net Market Income\n(In)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('DI', 'Disposable Income\n(Id)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('PFI', 'Post-fiscal Income\n(Ipf)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    center.node('FI', 'Final Income\n(If)', shape='ellipse', style='rounded,filled', fillcolor=color_income, fontcolor='white', width='3.8', height='1.3', fontsize='16', penwidth='1.3')
    # vertical ladder
    center.edge('MI', 'NMI', style='invis')
    center.edge('NMI', 'DI', style='invis')
    center.edge('DI', 'PFI', style='invis')
    center.edge('PFI', 'FI', style='invis')

# =====================================================
# RIGHT CLUSTER (OPERATIONS)
# =====================================================
with dot.subgraph(name='cluster_right') as right:
    right.attr(label='Income Modifications', style='rounded', color=color_add, fontcolor=color_add)
    right.node('DT', '(-) Direct Taxes ISR', shape='box', style='rounded,filled', fillcolor=color_sub, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('GT', '(+) Direct Government\nTransfers', shape='box', style='rounded,filled', fillcolor=color_add, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('IS', '(+) Indirect Subsidies\n(Electricity subsidy)', shape='box', style='rounded,filled', fillcolor=color_add, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('IT', '(-) Indirect Taxes\n(VAT / IEPS)', shape='box', style='rounded,filled', fillcolor=color_sub, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('IK', '(+) Education and Health\n(In-kind transfers)', shape='box', style='rounded,filled', fillcolor=color_add, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    right.node('CP', '(-) Copayments', shape='box', style='rounded,filled', fillcolor=color_sub, fontcolor='white', width='3.8', height='1.2', fontsize='16', penwidth='1.3')
    # vertical order
    right.edge('DT', 'GT', style='invis')
    right.edge('GT', 'IS', style='invis')
    right.edge('IS', 'IT', style='invis')
    right.edge('IT', 'IK', style='invis')
    right.edge('IK', 'CP', style='invis')

# =====================================================
# DATA SOURCES (PLACED BELOW EACH BOX, SHORT LINE)
# =====================================================
# Source for NMI
dot.node('src_NMI', 'Source. ENIGH', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='11', margin='0.2')
dot.edge('NMI', 'src_NMI', style='invis')
dot.edge('NMI', 'src_NMI', style='dotted', arrowhead='none', penwidth='0.8')

# Source for DT
dot.node('src_DT', 'Source. SAT', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='11', margin='0.2')
dot.edge('DT', 'src_DT', style='invis')
dot.edge('DT', 'src_DT', style='dotted', arrowhead='none', penwidth='0.8')

# Source for GT
dot.node('src_GT', 'Source. To define', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='11', margin='0.2')
dot.edge('GT', 'src_GT', style='invis')
dot.edge('GT', 'src_GT', style='dotted', arrowhead='none', penwidth='0.8')

# Source for IT
dot.node('src_IT', 'Source. To define', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='11', margin='0.2')
dot.edge('IT', 'src_IT', style='invis')
dot.edge('IT', 'src_IT', style='dotted', arrowhead='none', penwidth='0.8')

# Source for IK
dot.node('src_IK', 'Source. To define', shape='note', style='filled', fillcolor=color_source, fontcolor='white', fontsize='11', margin='0.2')
dot.edge('IK', 'src_IK', style='invis')
dot.edge('IK', 'src_IK', style='dotted', arrowhead='none', penwidth='0.8')

# =====================================================
# ORIGINAL CONNECTIONS (DEFINITIONS TO INCOME AND LADDER)
# =====================================================
dot.edge('def_MI', 'MI', style='dotted', arrowhead='none')
dot.edge('def_NMI', 'NMI', style='dotted', arrowhead='none')
dot.edge('def_DI', 'DI', style='dotted', arrowhead='none')
dot.edge('def_PFI', 'PFI', style='dotted', arrowhead='none')
dot.edge('def_FI', 'FI', style='dotted', arrowhead='none')

dot.edge('MI', 'DT', style='solid')
dot.edge('DT', 'NMI', style='solid')
dot.edge('NMI', 'GT', style='solid')
dot.edge('GT', 'DI', style='solid')
dot.edge('DI', 'IS', style='solid')
dot.edge('IS', 'IT', style='solid')
dot.edge('IT', 'PFI', style='solid')
dot.edge('PFI', 'IK', style='solid')
dot.edge('IK', 'CP', style='solid')
dot.edge('CP', 'FI', style='solid')

# =====================================================
# EXPORT: PDF AND PNG (FOR WORD INSERTION)
# =====================================================
dot.render('Metodologia_CEQ_Formato_Academico', format='pdf', cleanup=True)
dot.render('Metodologia_CEQ_Formato_Academico', format='png', cleanup=True)

print("✅ Diagram generated in PDF and PNG. You can insert the PNG file directly into Word.")

✅ Diagram generated in PDF and PNG. You can insert the PNG file directly into Word.
